#Exploring table 'players'

In [0]:
%sql
SELECT
  *
FROM tennislakehouse.bronze.players 
LIMIT 200

In [0]:
%sql
SELECT DISTINCT
  hand
FROM tennislakehouse.bronze.players

In [0]:
SELECT
  dob,
  LENGTH(CAST(CAST(dob AS BIGINT) AS STRING))
FROM tennislakehouse.bronze.players
WHERE LENGTH(CAST(CAST(dob AS BIGINT) AS STRING)) != 8
LIMIT 300
--no invalid dates detected

In [0]:
SELECT DISTINCT
  ioc
FROM tennislakehouse.bronze.players

In [0]:
SELECT
  *
FROM tennislakehouse.bronze.players
WHERE wikidata_id = 'Q108498952'

In [0]:
--exploring the duplicate wikidata players
SELECT 
  outliers.wikidata_id,
  outliers.player_id first_id,
  original.player_id second_id,
  outliers.name_first first_name_1,
  original.name_first first_name_2,
  outliers.name_last last_name_1,
  original.name_last last_name_2,
  outliers.hand,
  original.hand,
  outliers.dob,
  original.dob,
  outliers.ioc,
  original.ioc,
  outliers.height,
  original.height
FROM (
  SELECT 
    player_id,
    name_first,
    name_last,
    hand,
    dob,
    ioc,
    height, 
    wikidata_id
  FROM (
    SELECT
      *,
      ROW_NUMBER() OVER (PARTITION BY wikidata_id ORDER BY wikidata_id) flag
    FROM tennislakehouse.bronze.players
    WHERE wikidata_id IS NOT NULL
  )
  WHERE flag != 1
) outliers
LEFT JOIN tennislakehouse.bronze.players original
ON outliers.wikidata_id = original.wikidata_id
ORDER BY outliers.wikidata_id

In [0]:
SELECT
  *
FROM tennislakehouse.bronze.players
WHERE wikidata_id IN (
  SELECT 
    wikidata_id
  FROM (
    SELECT
      *,
      ROW_NUMBER() OVER (PARTITION BY wikidata_id ORDER BY wikidata_id) flag
    FROM tennislakehouse.bronze.players
    WHERE wikidata_id IS NOT NULL
  )
  WHERE flag != 1
)
ORDER BY wikidata_id

In [0]:
SELECT 
  COUNT(player_id) 
FROM tennislakehouse.bronze.players 
WHERE name_first IS NULL